# Baowerful — Colab training

Trains the CLIP + classifier-head AIGC detector on CIFAKE, using Colab's free GPU instead of your laptop.

**First:** go to `Runtime > Change runtime type` and pick a GPU (T4 is fine), then run the cells below top to bottom.

You'll need two things ready, neither of which is shared with anyone else — you paste/upload them directly into this session:
1. A GitHub **personal access token** (classic, `repo` scope) so this notebook can clone your team's repo and push results back. Create one at https://github.com/settings/tokens
2. Your Kaggle **API token** (`kaggle.json`) so this notebook can download the CIFAKE dataset directly, instead of you re-uploading the 469MB you already have locally. Get it at https://www.kaggle.com/settings -> "Create New Token"


In [ ]:
!nvidia-smi

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub personal access token: ")
repo_url = f"https://{token}@github.com/Tristan914684/Baowerful.git"

!git clone {repo_url} /content/repo
%cd /content/repo

# Kept in memory only for this session, used again later to push results back.
os.environ["GIT_TOKEN"] = token


In [ ]:
!pip install -q -r requirements.txt

Upload your `kaggle.json` below (the file Kaggle gives you when you create an API token).

In [ ]:
from google.colab import files
import os

print("Upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(list(uploaded.values())[0])
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip install -q kaggle


In [ ]:
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p /content/data_raw --unzip

In [ ]:
import shutil
from pathlib import Path

raw = Path("/content/data_raw")
dest = Path("/content/repo/data")

mapping = {
    ("train", "REAL"): dest / "train" / "real",
    ("train", "FAKE"): dest / "train" / "fake",
    ("test", "REAL"): dest / "val" / "real",
    ("test", "FAKE"): dest / "val" / "fake",
}

for (split, cls), target in mapping.items():
    src = raw / split / cls
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target)
    shutil.move(str(src), str(target))

for name in ["train/real", "train/fake", "val/real", "val/fake"]:
    print(name, len(list((dest / name).glob("*.jpg"))))


Quick smoke test on a small subset first — confirms the whole pipeline works before committing GPU time to a full run.

In [ ]:
!python -m src.train --train_dir data/train --val_dir data/val --epochs 1 --max_train_samples 500 --max_val_samples 200

If that printed accuracy numbers with no errors, run the real training below. Adjust `--epochs` as time allows — Colab's GPU makes each epoch much faster than a laptop CPU.

In [ ]:
!python -m src.train --train_dir data/train --val_dir data/val --epochs 10

Build the robustness evaluation table (clean vs. each transform) and error analysis examples — both required deliverables.

In [ ]:
!python -m src.eval_robustness --data_dir data/val --checkpoint results/head_best.pt

Push the robustness summary and error examples back to GitHub (small text files — safe to commit). The trained checkpoint itself is gitignored on purpose; download it directly instead in the next cell.

In [ ]:
!git config user.email "Lim-04@users.noreply.github.com"
!git config user.name "Lim-04"
!git add docs/robustness_summary.csv docs/error_examples.json
!git commit -m "Add robustness evaluation results from Colab run"
!git push https://{os.environ['GIT_TOKEN']}@github.com/Tristan914684/Baowerful.git


In [ ]:
from google.colab import files
files.download("results/head_best.pt")


Optional: try the required scoring script end-to-end on a folder of images (swap in your own folder path if you like).

In [ ]:
!python -m src.infer --image_dir data/val/fake --checkpoint results/head_best.pt --output /content/predictions_demo.json --batch_size 64